In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
import tiktoken

torch.manual_seed(42)   # 랜덤 초기화를 매번 같게 → 결과 재현 가능

@dataclass
class GPTConfig:
    block_size: int = 256     # 최대 context 길이 (wpe 행 수)
    vocab_size: int = 50257   # GPT-2 토크나이저 vocab (wte 행 수)
    n_layer: int = 6          # 블록 수
    n_head: int = 6           # 어텐션 헤드 수 (2-3에서 사용)
    n_embd: int = 384         # C

cfg = GPTConfig()
print(cfg)

GPTConfig(block_size=256, vocab_size=50257, n_layer=6, n_head=6, n_embd=384)


In [3]:
enc = tiktoken.get_encoding("gpt2")
texts = [
    "Once upon a time, there was a little girl named Lily.",
    "The cat sat on the mat and looked at the bird.",
]
token_lists = [enc.encode(t) for t in texts]
print([len(t) for t in token_lists])

T = min(len(t) for t in token_lists)          # 길이를 맞추기 위해 짧은 쪽에 맞춰 자름
idx = torch.tensor([t[:T] for t in token_lists])   # (B, T) 정수 텐서
print("idx:", idx.shape, idx.dtype)
print(idx)

[13, 12]
idx: torch.Size([2, 12]) torch.int64
tensor([[ 7454,  2402,   257,   640,    11,   612,   373,   257,  1310,  2576,
          3706, 20037],
        [  464,  3797,  3332,   319,   262,  2603,   290,  3114,   379,   262,
          6512,    13]])


In [4]:
wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)   # weight: (V, C)
wpe = nn.Embedding(cfg.block_size, cfg.n_embd)   # weight: (block_size, C)

B, T = idx.shape
assert T <= cfg.block_size, "context 길이 초과"

pos = torch.arange(T)          # (T,)    [0, 1, ..., T-1]
tok_emb = wte(idx)             # (B, T, C)  토큰 ID로 행 조회
pos_emb = wpe(pos)             # (T, C)     위치 번호로 행 조회
x = tok_emb + pos_emb          # (B, T, C)  브로드캐스팅: (T, C)가 배치마다 더해짐

print("tok_emb:", tok_emb.shape)
print("pos_emb:", pos_emb.shape)
print("x      :", x.shape)

# 검증 1: x[b, t] = wte[그 토큰] + wpe[t]
b, t = 1, 3
print(torch.allclose(x[b, t], wte.weight[idx[b, t]] + wpe.weight[t]))
# 검증 2: 같은 위치(t)면 배치가 달라도 같은 위치 벡터가 더해짐
print(torch.allclose(x[0, t] - tok_emb[0, t], x[1, t] - tok_emb[1, t]))

t = 3
d0 = x[0, t] - tok_emb[0, t]      # (C,) 문장 0에서 꺼낸 위치 성분
d1 = x[1, t] - tok_emb[1, t]      # (C,) 문장 1에서 꺼낸 위치 성분

print("최대 차이:", (d0 - d1).abs().max().item())          # 1e-7 수준이면 반올림 오차
print("허용 오차 1e-6:", torch.allclose(d0, d1, atol=1e-6))  # True
print("원본 비교:", torch.equal(pos_emb[t], wpe.weight[t]))   # True: 더한 위치 벡터 자체는 완전히 같음

tok_emb: torch.Size([2, 12, 384])
pos_emb: torch.Size([12, 384])
x      : torch.Size([2, 12, 384])
True
False
최대 차이: 4.76837158203125e-07
허용 오차 1e-6: True
원본 비교: True


In [5]:
B, T, C = x.shape                                # (2, 12, 384)

mask = torch.tril(torch.ones(T, T))
wei = torch.zeros(T, T)                          # (T, T) 점수가 전부 0 (=모든 토큰 동일 취급)
wei = wei.masked_fill(mask == 0, float("-inf"))  # 미래 자리를 -inf로
wei = F.softmax(wei, dim=-1)                     # (T, T) 각 행의 합 = 1
print(wei[:5, :5])

avg = wei @ x                                    # (T, T) @ (B, T, C) → (B, T, C)
# 검증: 2번 토큰의 결과 = 0~2번 토큰의 단순 평균
print(torch.allclose(avg[0, 2], x[0, :3].mean(dim=0), atol=1e-6))

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000]])
True


In [6]:
head_size = 64                                   # 2-3에서 헤드 6개 × 64 = 384가 됨

query = nn.Linear(C, head_size)                  # (C → hs)
key   = nn.Linear(C, head_size)
value = nn.Linear(C, head_size)

q = query(x)                                     # (B, T, hs)
k = key(x)                                       # (B, T, hs)
v = value(x)                                     # (B, T, hs)

# 1. 점수: 모든 토큰 쌍의 Q·K 내적
att = q @ k.transpose(-2, -1)                    # (B, T, hs) @ (B, hs, T) → (B, T, T)
att = att * head_size**-0.5                      # 스케일링 (셀 6에서 이유 확인)

# 2. 인과 마스크
att = att.masked_fill(mask == 0, float("-inf"))  # (B, T, T)

# 3. softmax → 가중치
att = F.softmax(att, dim=-1)                     # (B, T, T) 각 행 합 = 1

# 4. 가중합
out = att @ v                                    # (B, T, T) @ (B, T, hs) → (B, T, hs)

print("att:", att.shape, "/ out:", out.shape)
print(att[0, :5, :5].round(decimals=3))          # 첫 문장, 앞 5토큰의 가중치
print("행 합:", att[0].sum(dim=-1)[:5])

att: torch.Size([2, 12, 12]) / out: torch.Size([2, 12, 64])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7120, 0.2880, 0.0000, 0.0000, 0.0000],
        [0.3310, 0.1780, 0.4910, 0.0000, 0.0000],
        [0.4740, 0.2260, 0.1260, 0.1730, 0.0000],
        [0.3440, 0.0530, 0.1710, 0.3020, 0.1300]], grad_fn=<RoundBackward1>)
행 합: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SliceBackward0>)


In [7]:
torch.manual_seed(0)
for hs in [16, 64, 256]:
    qq = torch.randn(1000, hs)                   # 평균 0, 분산 1
    kk = torch.randn(1000, hs)
    dots = (qq * kk).sum(dim=-1)                 # 내적 1000개
    print(f"hs={hs:3d}  내적 분산: {dots.var():7.1f}  스케일 후: {(dots * hs**-0.5).var():.2f}")

scores = torch.tensor([1.0, 2.0, 3.0])
print("softmax(점수)     :", F.softmax(scores, dim=-1))
print("softmax(점수 × 8) :", F.softmax(scores * 8, dim=-1))

hs= 16  내적 분산:    16.0  스케일 후: 1.00
hs= 64  내적 분산:    63.1  스케일 후: 0.99
hs=256  내적 분산:   259.8  스케일 후: 1.02
softmax(점수)     : tensor([0.0900, 0.2447, 0.6652])
softmax(점수 × 8) : tensor([1.1250e-07, 3.3535e-04, 9.9966e-01])


In [8]:
n_head = cfg.n_head                     # 6
B, T, C = x.shape                       # (2, 12, 384)
hs = C // n_head                        # 64

c_attn = nn.Linear(C, 3 * C)            # (384 → 1152)
qkv = c_attn(x)                         # (B, T, 3C) = (2, 12, 1152)
q, k, v = qkv.split(C, dim=2)           # 마지막 차원을 384씩 3등분 → 각 (B, T, C)
print("qkv:", qkv.shape, "→ q, k, v:", q.shape)

qkv: torch.Size([2, 12, 1152]) → q, k, v: torch.Size([2, 12, 384])


In [9]:
q = q.view(B, T, n_head, hs).transpose(1, 2)   # (B, T, C) → (B, T, nh, hs) → (B, nh, T, hs)
k = k.view(B, T, n_head, hs).transpose(1, 2)   # (B, nh, T, hs)
v = v.view(B, T, n_head, hs).transpose(1, 2)   # (B, nh, T, hs)
print("q:", q.shape)

q: torch.Size([2, 6, 12, 64])


In [10]:
att = q @ k.transpose(-2, -1) * hs**-0.5        # (B, nh, T, hs) @ (B, nh, hs, T) → (B, nh, T, T)
att = att.masked_fill(mask == 0, float("-inf")) # mask (T, T)가 (B, nh)마다 브로드캐스팅
att = F.softmax(att, dim=-1)                    # (B, nh, T, T)
y = att @ v                                     # (B, nh, T, T) @ (B, nh, T, hs) → (B, nh, T, hs)

print("att:", att.shape, "/ y:", y.shape)
print("헤드 0:\n", att[0, 0, :4, :4].round(decimals=2))
print("헤드 1:\n", att[0, 1, :4, :4].round(decimals=2))

att: torch.Size([2, 6, 12, 12]) / y: torch.Size([2, 6, 12, 64])
헤드 0:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.6200, 0.0000, 0.0000],
        [0.5400, 0.1800, 0.2900, 0.0000],
        [0.4700, 0.2300, 0.2000, 0.1000]], grad_fn=<RoundBackward1>)
헤드 1:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.6100, 0.3900, 0.0000, 0.0000],
        [0.3100, 0.3900, 0.2900, 0.0000],
        [0.0900, 0.6800, 0.0900, 0.1400]], grad_fn=<RoundBackward1>)


In [11]:
y = y.transpose(1, 2).contiguous().view(B, T, C)   # (B, nh, T, hs) → (B, T, nh, hs) → (B, T, C)
c_proj = nn.Linear(C, C)                           # (384 → 384)
out = c_proj(y)                                    # (B, T, C)
print("out:", out.shape)

n_params = sum(p.numel() for p in c_attn.parameters()) + sum(p.numel() for p in c_proj.parameters())
print(f"어텐션 파라미터: {n_params:,}  (4C² = {4 * C**2:,})")

out: torch.Size([2, 12, 384])
어텐션 파라미터: 591,360  (4C² = 589,824)


In [12]:
q2, k2, v2 = [t.view(B, T, n_head, hs).transpose(1, 2) for t in c_attn(x).split(C, dim=2)]
y_sdpa = F.scaled_dot_product_attention(q2, k2, v2, is_causal=True)   # (B, nh, T, hs)
y_mine = att @ v                                                      # 셀 9 결과

print("최대 차이:", (y_sdpa - y_mine).abs().max().item())
print(torch.allclose(y_sdpa, y_mine, atol=1e-5))

최대 차이: 2.384185791015625e-07
True


In [13]:
class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.c_attn = nn.Linear(cfg.n_embd, 3 * cfg.n_embd)   # Q, K, V 한꺼번에 (GPT-2 이름)
        self.c_proj = nn.Linear(cfg.n_embd, cfg.n_embd)       # 헤드 합친 뒤 출력 (GPT-2 이름)
        self.n_head = cfg.n_head
        self.n_embd = cfg.n_embd

    def forward(self, x):
        B, T, C = x.size()
        hs = C // self.n_head
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)       # 각 (B, T, C)
        q = q.view(B, T, self.n_head, hs).transpose(1, 2)        # (B, nh, T, hs)
        k = k.view(B, T, self.n_head, hs).transpose(1, 2)        # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, hs).transpose(1, 2)        # (B, nh, T, hs)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)  # (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C)         # (B, T, C) 헤드 합치기
        return self.c_proj(y)                                    # (B, T, C)

attn = CausalSelfAttention(cfg)
out = attn(x)
print("out:", out.shape)
for name, p in attn.named_parameters():
    print(f"{name:15s} {tuple(p.shape)}")

W = attn.c_attn.weight          # (1152, 384)  우리 방식 (출력, 입력)
b = attn.c_attn.bias            # (1152,)
W_hf = W.T                      # (384, 1152)  HF 방식 (입력, 출력)

y1 = attn.c_attn(x)             # nn.Linear 계산
y2 = x @ W_hf + b               # Conv1D 방식 계산
print(torch.allclose(y1, y2, atol=1e-5))

out: torch.Size([2, 12, 384])
c_attn.weight   (1152, 384)
c_attn.bias     (1152,)
c_proj.weight   (384, 384)
c_proj.bias     (384,)
True


In [14]:
ln = nn.LayerNorm(cfg.n_embd)                  # weight (C,), bias (C,)

h = out                                        # 2-3 어텐션 출력 (B, T, C)
print("정규화 전  평균:", h[0, 0].mean().item(), " 표준편차:", h[0, 0].std().item())
y_ln = ln(h)                                   # (B, T, C) 토큰마다 384개를 정규화
print("정규화 후  평균:", y_ln[0, 0].mean().item(), " 표준편차:", y_ln[0, 0].std().item())
print(y_ln.shape)

정규화 전  평균: 0.0513155572116375  표준편차: 0.4577833116054535
정규화 후  평균: 3.725290298461914e-09  표준편차: 1.0012807846069336
torch.Size([2, 12, 384])


In [15]:
class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.c_fc   = nn.Linear(cfg.n_embd, 4 * cfg.n_embd)    # 384 → 1536 (GPT-2 이름)
        self.gelu   = nn.GELU(approximate="tanh")              # GPT-2와 같은 근사 버전
        self.c_proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd)    # 1536 → 384 (GPT-2 이름)

    def forward(self, x):
        x = self.c_fc(x)        # (B, T, 4C)
        x = self.gelu(x)        # (B, T, 4C)
        x = self.c_proj(x)      # (B, T, C)
        return x

mlp = MLP(cfg)
print("mlp out:", mlp(x).shape)
print(f"MLP 파라미터: {sum(p.numel() for p in mlp.parameters()):,}  (8C² = {8 * cfg.n_embd**2:,})")

mlp out: torch.Size([2, 12, 384])
MLP 파라미터: 1,181,568  (8C² = 1,179,648)


In [16]:
class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd)
        self.mlp  = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))   # (B, T, C)  정규화 → 어텐션 → 원래 x에 더하기
        x = x + self.mlp(self.ln_2(x))    # (B, T, C)  정규화 → MLP   → 원래 x에 더하기
        return x

block = Block(cfg)
print("block out:", block(x).shape)
for name, p in block.named_parameters():
    print(f"{name:22s} {tuple(p.shape)}")
n_block = sum(p.numel() for p in block.parameters())
print(f"블록 파라미터: {n_block:,}  (12C² = {12 * cfg.n_embd**2:,})")

block out: torch.Size([2, 12, 384])
ln_1.weight            (384,)
ln_1.bias              (384,)
attn.c_attn.weight     (1152, 384)
attn.c_attn.bias       (1152,)
attn.c_proj.weight     (384, 384)
attn.c_proj.bias       (384,)
ln_2.weight            (384,)
ln_2.bias              (384,)
mlp.c_fc.weight        (1536, 384)
mlp.c_fc.bias          (1536,)
mlp.c_proj.weight      (384, 1536)
mlp.c_proj.bias        (384,)
블록 파라미터: 1,774,464  (12C² = 1,769,472)


In [19]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("..")             # notebooks/의 상위 폴더(mini-llm)를 import 경로에 추가

from src.model import GPT, GPTConfig

model = GPT(GPTConfig())
logits, loss = model(idx)         # idx: 셀 2의 (2, 12)
print("logits:", logits.shape)

# weight tying 확인: 두 이름이 같은 메모리를 가리키는가
print("tied:", model.lm_head.weight is model.transformer.wte.weight)

n_params = sum(p.numel() for p in model.parameters())   # 공유 파라미터는 한 번만 셈
print(f"전체 파라미터: {n_params:,}")

for name, p in list(model.named_parameters())[:6]:
    print(f"{name:35s} {tuple(p.shape)}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
logits: torch.Size([2, 12, 50257])
tied: True
전체 파라미터: 30,044,544
transformer.wte.weight              (50257, 384)
transformer.wpe.weight              (256, 384)
transformer.h.0.ln_1.weight         (384,)
transformer.h.0.ln_1.bias           (384,)
transformer.h.0.attn.c_attn.weight  (1152, 384)
transformer.h.0.attn.c_attn.bias    (1152,)


In [ ]:
import math

x_in  = idx[:, :-1]          # (B, T-1) 입력: 0 ~ 10번 토큰
y_tgt = idx[:, 1:]           # (B, T-1) 정답: 1 ~ 11번 토큰 (한 칸씩 밀린 것)

logits, loss = model(x_in, y_tgt)
print(f"초기 loss: {loss.item():.3f}   기대값 ln(50257) = {math.log(50257):.3f}")

probs = F.softmax(logits[0, -1], dim=-1)    # (V,) 마지막 위치의 확률 분포
print(f"확률 최대 {probs.max().item():.2e} / 최소 {probs.min().item():.2e} / 균등 1/V = {1/50257:.2e}")

RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.